# Video Preprocessing for Reduced Order Modelling

Before any data-driven analysis can begin, raw experimental or simulation data must be brought into a form that numerical methods can work with. For video-based fluid flow data this means:

1. **Resizing** — standardising the spatial resolution so that all frames share the same pixel dimensions,
2. **Frame extraction** — converting the video stream into a NumPy array of shape `(N_t, H, W, C)`,
3. **Grayscale conversion** — reducing each RGB frame to a single intensity channel, giving shape `(N_t, H, W)`.

The resulting array is the **snapshot matrix** — the primary input to POD/SVD analysis and neural-network time-stepping. Keeping the resolution moderate (256 × 128) balances spatial fidelity against computational cost: SVD on a $32768 \times N_t$ matrix is already expensive; a larger resolution would make it prohibitive without changing the physical content.

The video used here is a simulation of the **von Kármán vortex street** — the periodic shedding of counter-rotating vortices behind a cylindrical obstacle — a canonical benchmark for reduced-order modelling.

## 1. Background: Data Layout for ROM

Reduced-order methods operate on the **snapshot matrix** $\mathbf{X} \in \mathbb{R}^{N_x \times N_t}$, where:

- $N_x = H \times W$ is the number of spatial degrees of freedom (pixels),
- $N_t$ is the number of time snapshots (frames).

Each column $\mathbf{x}_k \in \mathbb{R}^{N_x}$ is a **flattened frame** at time $t_k$:

$$\mathbf{X} = \begin{bmatrix} | & | & & | \\ \mathbf{x}_1 & \mathbf{x}_2 & \cdots & \mathbf{x}_{N_t} \\ | & | & & | \end{bmatrix}$$

Grayscale intensity values are real numbers in $[0, 255]$ (uint8) or $[0, 1]$ after normalisation. For SVD we use the raw uint8 values — the decomposition is scale-invariant up to a global constant.

### Why 256 × 128?

The original video has a 2:1 aspect ratio (width:height). Downsampling to 256 × 128 preserves this ratio while keeping $N_x = 32{,}768$ — small enough that a full SVD completes in seconds on a laptop.

## 2. Imports

## 3. Inspecting the Raw Video

Before processing, we read the video metadata — frame count, frame rate, and native resolution — to understand what we are working with and to inform the choice of target resolution.

## 4. Resizing the Video

We resize every frame to **256 × 128** and write the result to `output.mp4`. The function below is a reusable utility: it reads the source video frame by frame, resizes each frame with `cv2.resize`, and writes it to a new file at the same frame rate.

## 5. Extracting Frames

We load the resized video and collect every frame into a Python list, then convert to a NumPy array of shape `(N_t, H, W, 3)`. This colour array is saved as `frames_array_color.npy` before grayscale conversion.

## 6. Converting to Grayscale

POD treats each pixel as a scalar state variable. Using three colour channels would triple the dimensionality without adding meaningful information for this flow — the dynamics are captured in intensity variations, not colour. We therefore convert each BGR frame to a single luminance channel:

$$I = 0.114\,B + 0.587\,G + 0.299\,R$$

This is the standard ITU-R BT.601 luminance formula applied by `cv2.COLOR_BGR2GRAY`.

## 7. Visualising Sample Frames

A quick visual check confirms that the preprocessing pipeline produced sensible output: uniform resolution, correct aspect ratio, and the characteristic vortex-shedding pattern visible even in grayscale.

## 8. Summary and Key Takeaways

| Step | Input | Output | Shape |
|------|-------|--------|-------|
| Resize | `karman.mp4` (native res.) | `output.mp4` | — |
| Extract | `output.mp4` | `frames_array_color.npy` | $(N_t, 128, 256, 3)$ |
| Grayscale | color array | `frames_array.npy` | $(N_t, 128, 256)$ |

### Key takeaways

- **Consistent resolution** is mandatory before stacking frames into a snapshot matrix — SVD requires all columns to have the same length.
- **Grayscale conversion** reduces dimensionality by $3\times$ with negligible information loss for intensity-driven flows.
- The output file `frames_array.npy` is the direct input to the `POD` and `TimeSteppingNN` notebooks.